# 01 · Fine-tuning SegFormer-B0 untuk Segmentasi Trotoar

Model ini menghasilkan **mask trotoar** (2 kelas: background, sidewalk) yang dipakai pipeline geolokasi (`03`) untuk dua hal: (1) memfilter obstacle yang benar-benar berada di trotoar, dan (2) menyediakan piksel permukaan tanah untuk fit bidang (RANSAC).

**Output:**
- `model_finetune_segformer/modelv1/segformer_final.ckpt` → checkpoint epoch terakhir. **Dipakai notebook `03`** dan dipublikasikan di [Hugging Face](https://huggingface.co/wahyunan/sidewalk-obstacle-geolocation).
- `model_finetune_segformer/modelv1/segformer-epoch=XX-val_loss=YY.ckpt` → checkpoint val_loss terendah (pembanding)
- `model_finetune_segformer/modelv1/logs/` → metrik per epoch (CSV)

**Ringkasan run terakhir (Sept 2025):** best val_loss 0.076 di epoch 10; early stopping 10 epoch kemudian.

**Struktur:** Bagian 1–4 setup dan data · Bagian 5 model · Bagian 6 training (*jangan dijalankan ulang kecuali memang mau retrain*) · Bagian 7 evaluasi test set (bisa langsung dijalankan dari checkpoint di Drive) · Bagian 8 contoh prediksi.

## 1. Setup

In [ ]:
# transformers dipin sama dengan notebook inference (03): struktur layer SegFormer berubah di transformers 5.x,
# sehingga checkpoint harus dilatih dan di-load dengan major version yang sama.
!pip install -q "transformers==4.57.6" pytorch-lightning torchmetrics roboflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from torchmetrics import JaccardIndex, Accuracy
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

pl.seed_everything(42)   # reproducibility: shuffle dataloader dan inisialisasi kepala klasifikasi
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Dataset (Roboflow)

Dataset `sidewalk-zhrul` versi 2, diekspor dalam format **png-mask-semantic**: setiap gambar `.jpg` punya mask `.png` dengan nilai piksel = id kelas, plus `_classes.csv` berisi daftar kelas.

In [ ]:
from roboflow import Roboflow

# API key Roboflow dibaca dari Colab Secrets (ikon kunci di sidebar kiri), bukan ditulis di notebook.
# Tambahkan secret bernama ROBOFLOW_API_KEY dan aktifkan "Notebook access".
from google.colab import userdata
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

project = rf.workspace("dika-biyq4").project("sidewalk-zhrul")
dataset = project.version(2).download("png-mask-semantic")

In [ ]:
for split in ["train", "valid", "test"]:
    d = os.path.join(dataset.location, split)
    n_img = len([f for f in os.listdir(d) if f.endswith(".jpg")])
    print(f"{split:5s}: {n_img} gambar")
print(open(os.path.join(dataset.location, "train", "_classes.csv")).read())

## 3. Dataset class

Gambar dan mask dipasangkan berdasarkan urutan nama file (Roboflow memberi nama yang sama untuk keduanya, beda akhiran `_mask`).

Image processor me-*resize* gambar dan mask ke 512×512 dan menormalisasi gambar dengan mean/std ImageNet (sesuai pretraining SegFormer). **Tidak ada augmentasi**. Ini salah satu hal yang bisa ditambahkan kalau mau membagusin model.

In [ ]:
class SemanticSegmentationDataset(Dataset):
    def __init__(self, root_dir, image_processor):
        self.root_dir = root_dir
        self.image_processor = image_processor
        with open(os.path.join(root_dir, "_classes.csv")) as fid:
            data = [l.strip().split(",") for i, l in enumerate(fid) if i != 0]
        self.id2label = {int(x[0]): x[1].strip() for x in data}
        self.images = sorted(f for f in os.listdir(root_dir) if f.endswith(".jpg"))
        self.masks = sorted(f for f in os.listdir(root_dir) if f.endswith(".png"))
        assert len(self.images) == len(self.masks), "jumlah gambar dan mask tidak sama"

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(os.path.join(self.root_dir, self.images[idx]))
        seg_map = Image.open(os.path.join(self.root_dir, self.masks[idx]))
        enc = self.image_processor(image, seg_map, return_tensors="pt")   # resize 512 + normalisasi
        return {k: v.squeeze(0) for k, v in enc.items()}                  # buang dimensi batch

## 4. Image processor dan DataLoader

| Parameter | Nilai | Dasar |
|---|---|---|
| `size` | 512×512 | Resolusi pretraining checkpoint `segformer-b0-finetuned-ade-512-512`. Di bawahnya (mis. 128) detail tepi trotoar hilang; di atasnya memori GPU naik kuadratik. |
| `do_reduce_labels` | `False` | `True` menggeser label −1 dan menjadikan 0 sebagai "ignore" (konvensi ADE20K). Di dataset ini 0 = background yang valid, jadi harus `False`. Notebook inference memakai setting yang sama. |
| `batch_size` | 8 | Muat di GPU Colab untuk 512×512 dengan mixed precision. |

In [ ]:
image_processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
image_processor.do_reduce_labels = False                 # 0 = background yang valid, bukan "ignore"
image_processor.size = {"height": 512, "width": 512}     # resolusi pretraining

train_dataset = SemanticSegmentationDataset(f"{dataset.location}/train/", image_processor)
val_dataset   = SemanticSegmentationDataset(f"{dataset.location}/valid/", image_processor)
test_dataset  = SemanticSegmentationDataset(f"{dataset.location}/test/",  image_processor)
ID2LABEL = train_dataset.id2label
print("Kelas:", ID2LABEL)

batch_size, num_workers = 8, 2
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_dataloader   = DataLoader(val_dataset,   batch_size=batch_size, num_workers=num_workers)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, num_workers=num_workers)

## 5. Model (LightningModule)

- **Backbone:** `nvidia/segformer-b0-finetuned-ade-512-512` (MiT-B0, 3.7 juta parameter). Varian terkecil SegFormer, cepat untuk inferensi foto beresolusi tinggi. Kepala klasifikasi 150 kelas ADE20K diganti 2 kelas (`ignore_mismatched_sizes=True`); bobot backbone tetap dari pretraining.
- **Metrik:** mean IoU (Jaccard) dan pixel accuracy dari `torchmetrics`. `ignore_index=255` untuk piksel tanpa label.
- **Optimizer:** AdamW, lr 2e-5. Lebih kecil dari 6e-5 yang dipakai di paper SegFormer (Xie et al., 2021) karena ini fine-tuning dari bobot yang sudah dilatih untuk segmentasi; lr kecil mengurangi risiko merusak fitur pretrained.

> Class ini **harus sama** (nama dan argumen `__init__`) dengan yang ada di notebook `03`, supaya checkpoint Lightning bisa di-load.

In [ ]:
class SegformerFinetuner(pl.LightningModule):
    def __init__(self, id2label, train_dataloader=None, val_dataloader=None, test_dataloader=None,
                 metrics_interval=100):
        super().__init__()
        self.id2label = id2label
        self.metrics_interval = metrics_interval
        self.train_dl, self.val_dl, self.test_dl = train_dataloader, val_dataloader, test_dataloader
        self.num_classes = len(id2label)
        self.label2id = {v: k for k, v in id2label.items()}

        self.model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/segformer-b0-finetuned-ade-512-512",
            return_dict=False,
            num_labels=self.num_classes,
            id2label=self.id2label,
            label2id=self.label2id,
            ignore_mismatched_sizes=True,      # kepala 150 kelas -> 2 kelas, backbone tetap pretrained
        )

        kw = dict(task="multiclass", num_classes=self.num_classes, ignore_index=255)
        self.train_mean_iou, self.val_mean_iou, self.test_mean_iou = JaccardIndex(**kw), JaccardIndex(**kw), JaccardIndex(**kw)
        self.train_accuracy, self.val_accuracy, self.test_accuracy = Accuracy(**kw), Accuracy(**kw), Accuracy(**kw)

    def forward(self, images, masks):
        return self.model(pixel_values=images, labels=masks)

    def _shared_step(self, batch):
        images, masks = batch["pixel_values"], batch["labels"]
        loss, logits = self(images, masks)[:2]
        # logits SegFormer berukuran 1/4 input -> di-upsample ke ukuran mask sebelum argmax
        up = nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        return loss, up.argmax(dim=1), masks

    def training_step(self, batch, batch_idx):
        loss, pred, masks = self._shared_step(batch)
        self.train_mean_iou.update(pred, masks)
        self.train_accuracy.update(pred, masks)
        if batch_idx % self.metrics_interval == 0:
            self.log("train_loss", loss, prog_bar=True)
            self.log("train_mean_iou", self.train_mean_iou.compute(), prog_bar=True)
            self.log("train_accuracy", self.train_accuracy.compute(), prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        # reset tiap epoch supaya metrik train tidak terakumulasi lintas epoch
        self.train_mean_iou.reset()
        self.train_accuracy.reset()

    def validation_step(self, batch, batch_idx):
        loss, pred, masks = self._shared_step(batch)
        self.val_mean_iou.update(pred, masks)
        self.val_accuracy.update(pred, masks)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        self.log("val_mean_iou", self.val_mean_iou.compute(), prog_bar=True)
        self.log("val_accuracy", self.val_accuracy.compute(), prog_bar=True)
        self.val_mean_iou.reset()
        self.val_accuracy.reset()

    def test_step(self, batch, batch_idx):
        loss, pred, masks = self._shared_step(batch)
        self.test_mean_iou.update(pred, masks)
        self.test_accuracy.update(pred, masks)
        self.log("test_loss", loss, prog_bar=True)
        return loss

    def on_test_epoch_end(self):
        self.log("test_mean_iou", self.test_mean_iou.compute(), prog_bar=True)
        self.log("test_accuracy", self.test_accuracy.compute(), prog_bar=True)
        self.test_mean_iou.reset()
        self.test_accuracy.reset()

    def configure_optimizers(self):
        return torch.optim.AdamW([p for p in self.parameters() if p.requires_grad], lr=2e-5, eps=1e-8)

    def train_dataloader(self):
        return self.train_dl

    def val_dataloader(self):
        return self.val_dl

    def test_dataloader(self):
        return self.test_dl

## 6. Training

> **Jangan dijalankan ulang** kecuali memang mau retrain. Checkpoint hasil run sebelumnya sudah ada di Drive, dan Bagian 7–8 bisa langsung dijalankan.

| Parameter | Nilai | Dasar |
|---|---|---|
| `EarlyStopping(monitor="val_loss", patience=10)` | 10 epoch | Berhenti kalau val_loss tidak membaik 10 epoch berturut-turut. Run terakhir: terbaik di epoch 10, berhenti di epoch 20. |
| `ModelCheckpoint(save_top_k=1, monitor="val_loss")` | 1 terbaik | Menyimpan checkpoint dengan val_loss terendah sebagai pembanding. Notebook `03` memakai `segformer_final.ckpt` (epoch terakhir), yang sudah divalidasi end-to-end; Bagian 7 membandingkan IoU keduanya. |
| `max_epochs` | 500 | Batas atas saja; training praktis dihentikan oleh early stopping. |
| `precision="16-mixed"` | FP16 | Hemat memori dan ~2× lebih cepat di GPU Colab, tanpa penurunan akurasi yang berarti untuk fine-tuning. |
| `CSVLogger` | `logs/` | Menyimpan metrik per epoch (val_loss, val_mean_iou) ke Drive untuk dokumentasi. Baru ada kalau training dijalankan ulang. |

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/model_finetune_segformer/modelv1/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

segformer_finetuner = SegformerFinetuner(
    ID2LABEL,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    test_dataloader=test_dataloader,
    metrics_interval=10,
)

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=0.0, patience=10, verbose=True, mode="min")
checkpoint_callback = ModelCheckpoint(dirpath=CHECKPOINT_DIR, filename="segformer-{epoch:02d}-{val_loss:.2f}",
                                      save_top_k=1, monitor="val_loss", mode="min")

trainer = Trainer(
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    callbacks=[early_stop_callback, checkpoint_callback],
    logger=CSVLogger(CHECKPOINT_DIR, name="logs"),
    max_epochs=500,
    check_val_every_n_epoch=1,
    log_every_n_steps=10,
)
trainer.fit(segformer_finetuner)

final_ckpt_path = os.path.join(CHECKPOINT_DIR, "segformer_final.ckpt")
trainer.save_checkpoint(final_ckpt_path)
print("Best checkpoint :", checkpoint_callback.best_model_path, f"(val_loss {checkpoint_callback.best_model_score:.4f})")
print("Final checkpoint:", final_ckpt_path)

In [ ]:
# Kurva training (hanya ada kalau training dijalankan dengan CSVLogger)
logs = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "logs", "version_*", "metrics.csv")))
if logs:
    m = pd.read_csv(logs[-1])
    ep = m.dropna(subset=["val_loss"]).groupby("epoch").last()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ep["val_loss"].plot(ax=axes[0], marker="o", title="val_loss")
    ep["val_mean_iou"].plot(ax=axes[1], marker="o", title="val_mean_iou")
    for a in axes: a.grid(True)
    plt.tight_layout(); plt.show()
else:
    print("Belum ada log CSV (training lama belum memakai CSVLogger).")

## 7. Evaluasi di test set

Bisa dijalankan tanpa training ulang: memuat checkpoint dari Drive, lalu menghitung **IoU per kelas** dan mIoU di test set. Test set belum pernah dievaluasi di run sebelumnya, jadi angka ini yang dilaporkan di dokumentasi.

IoU sidewalk adalah angka yang paling relevan untuk pipeline geolokasi, karena mask sidewalk dipakai untuk filter obstacle dan fit bidang tanah.

In [ ]:
CHECKPOINT_DIR = "/content/drive/MyDrive/model_finetune_segformer/modelv1/"
best = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "segformer-epoch=*.ckpt")))
CKPTS = {"best": best[-1] if best else None,
         "final": os.path.join(CHECKPOINT_DIR, "segformer_final.ckpt")}
print(CKPTS)


@torch.no_grad()
def evaluate(ckpt_path, dataloader, num_classes):
    model = SegformerFinetuner.load_from_checkpoint(ckpt_path, id2label=ID2LABEL).eval().to(device)
    conf = torch.zeros(num_classes, num_classes, dtype=torch.long)   # confusion matrix [gt, pred]
    for batch in dataloader:
        images, masks = batch["pixel_values"].to(device), batch["labels"]
        logits = model.model(pixel_values=images)[0]
        pred = nn.functional.interpolate(logits, size=masks.shape[-2:], mode="bilinear",
                                         align_corners=False).argmax(1).cpu()
        valid = masks != 255
        conf += torch.bincount(masks[valid] * num_classes + pred[valid],
                               minlength=num_classes ** 2).reshape(num_classes, num_classes)
    tp = conf.diag().float()
    iou = tp / (conf.sum(0) + conf.sum(1) - tp).float()
    acc = tp.sum() / conf.sum()
    return iou.numpy(), float(acc)


rows = []
for name, path in CKPTS.items():
    if path and os.path.exists(path):
        iou, acc = evaluate(path, test_dataloader, len(ID2LABEL))
        row = {"checkpoint": name, "file": os.path.basename(path)}
        row.update({f"IoU_{ID2LABEL[i]}": iou[i] for i in range(len(iou))})
        row.update(mIoU=iou.mean(), pixel_acc=acc)
        rows.append(row)
eval_df = pd.DataFrame(rows)
eval_df.round(4)

## 8. Contoh prediksi di test set

Hijau = prediksi sidewalk. Kolom kanan menunjukkan kesalahan: merah = false positive (diprediksi sidewalk padahal bukan), biru = false negative (sidewalk yang terlewat).

In [ ]:
CKPT_FOR_VIS = CKPTS["final"]   # checkpoint yang dipakai pipeline (notebook 03)
vis_model = SegformerFinetuner.load_from_checkpoint(CKPT_FOR_VIS, id2label=ID2LABEL).eval().to(device)
SIDEWALK_ID = [k for k, v in ID2LABEL.items() if "side" in v.lower()][0]

idxs = np.random.default_rng(0).choice(len(test_dataset), min(4, len(test_dataset)), replace=False)
fig, axes = plt.subplots(len(idxs), 3, figsize=(15, 4 * len(idxs)))
axes = np.atleast_2d(axes)
for row, i in zip(axes, idxs):
    img = np.array(Image.open(os.path.join(test_dataset.root_dir, test_dataset.images[i])).convert("RGB"))
    gt = np.array(Image.open(os.path.join(test_dataset.root_dir, test_dataset.masks[i])))
    enc = image_processor(images=Image.fromarray(img), return_tensors="pt")
    with torch.no_grad():
        logits = vis_model.model(pixel_values=enc["pixel_values"].to(device))[0]
    pred = nn.functional.interpolate(logits, size=gt.shape, mode="bilinear",
                                     align_corners=False).argmax(1)[0].cpu().numpy()
    p, g = pred == SIDEWALK_ID, gt == SIDEWALK_ID
    ov = img.copy(); ov[p] = (0.5 * ov[p] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    err = img.copy() // 2
    err[p & ~g] = [255, 0, 0]; err[~p & g] = [0, 0, 255]
    for a, im, t in zip(row, [img, ov, err], ["Gambar", "Prediksi sidewalk", "Error (merah FP, biru FN)"]):
        a.imshow(im); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

## Catatan untuk pengembangan

- Notebook `03` memakai `segformer_final.ckpt` (epoch terakhir), yang sudah divalidasi end-to-end. Checkpoint val_loss terendah (`segformer-epoch=10-...ckpt`) bisa dibandingkan lewat Bagian 7 sebelum dipertimbangkan sebagai pengganti.
- **Augmentasi** (flip horizontal, perubahan brightness/kontras, random crop) belum dipakai; berguna untuk variasi pencahayaan, termasuk area teduh di bawah pohon.
- Backbone lebih besar (`segformer-b1`/`b2`) bisa meningkatkan IoU dengan biaya inferensi lebih lambat.

Referensi: Xie et al. (2021), *SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers*, NeurIPS. https://arxiv.org/abs/2105.15203